<a href="https://colab.research.google.com/github/khine-thant-su/crisis_companion_chatbot/blob/main/Layers/candidate_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs code suggested by ChatGPT to generate multiple diverse candidate replies for a single user message.

It uses 3 different models (distilgpt2, gpt2, and EleutherAI/gpt-neo-125M) to generate different replies, using Google Colab GPU.

In [2]:
# Run this once in a fresh Colab runtime.
!pip install -q transformers accelerate torch  # ! tells Colab to execute the command as a shell command rather than as Python code

# NOTE: Colab already has torch; the pip install ensures compatible versions.

In [3]:
# Check GPU availability & show device
import torch
gpu_avail = torch.cuda.is_available()
device = "cuda" if gpu_avail else "cpu"
print("GPU available?", gpu_avail)
print("Using device:", device)

# Optional: show GPU name if available
if gpu_avail:
    try:
        !nvidia-smi -L  # A shell command to list all NVIDIA GPUs in the system. This displays the GPU's details.
    except Exception:
        pass

GPU available? True
Using device: cuda
GPU 0: Tesla T4 (UUID: GPU-8652b4f4-498e-cc9d-4097-9b938b210feb)


In [4]:
# Imports and configuration
from typing import List, Dict, Any
import time, hashlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from functools import lru_cache

# Use GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Short system prompt to guide small models towards safe/empathic replies.
SYSTEM_PREFIX = (
    "You are an empathetic, non-clinical mental health support assistant. "
    "Do NOT offer medical or legal advice. Be supportive, validate feelings, and offer low-risk coping suggestions."
)

# Keep responses reasonably short on purpose
MAX_NEW_TOKENS = 120

In [5]:
# Lightweight cache / loader (helps when iterating)
@lru_cache(maxsize=8)
def load_model_and_tokenizer(model_name: str, device: str = DEVICE):
    """
    Load & return (model, tokenizer).
    Uses caching so repeated calls are fast while developing.
    """
    print(f"[loader] Loading {model_name} -> device={device} (this may take a moment)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Small models sometimes lack pad token — set it to eos_token to avoid issues (eos_token = end-of-sequence token --> special token added to end of a sequence of tokens to signal the model that the sequence is complete)
    # Note that the model's behavior with eos_token as padding might be slightly different than with a true pad_token.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load the model. For Colab GPU we can try to use half-precision for the weights and activations of the model for memory savings.
    # GPUs are efficient at parallel processing, but have a limited amount of memory. Using half-precision allows larger models to fit into GPU memory, or allows for larger batch sizes, which can speed up training.
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.to(device)
    # try converting to fp16 when on CUDA — memory saver (safe for inference)
    if device == "cuda":
        try:
            model.half()  # Convert the model's parameters to half-precision
        except Exception:
            # if half() fails for some backbone/config, it's fine to continue in fp32
            pass
    model.eval()
    print(f"[loader] {model_name} ready.")
    return model, tokenizer

In [6]:
# Core candidate-generation function
# Paste this into your Colab (replaces previous generation function)
import re
from typing import List, Dict, Any

# a small, conservative list of substrings that should never be returned to users.
# This is a defensive, temporary *dev* measure. Proper safety should be done with classifiers & human review.
_HARMFUL_WORDS = [
    "kill myself", "kill myself.", "kill myself!", "i will kill myself",
    "i'm going to kill myself", "i am going to kill myself", "i will kill myself soon",
    "i will kill you", "suicide", "die by suicide", "end my life", "i want to die", "im going to die"
]
_HARMFUL_RE = re.compile("|".join(re.escape(w) for w in _HARMFUL_WORDS), flags=re.IGNORECASE)

def _quick_hard_fail_check(text: str) -> bool:
    """Return True if the text contains disallowed substrings (very conservative)."""
    return bool(_HARMFUL_RE.search(text))


# Fixed generator: avoid bracket echoes + add repetition controls
def generate_candidates_for_message_no_echo(
    user_message: str,
    generation_specs: List[Dict[str, Any]],
    base_system_prefix: str = SYSTEM_PREFIX,
) -> List[Dict[str, Any]]:
    """
    Improved generator:
      - merges prompt_style into system instructions (no bracket tokens)
      - uses repetition_penalty and no_repeat_ngram_size
      - decodes only continuation tokens
    """
    candidates = []
    for spec in generation_specs:
        model_name = spec.get("model_name", "gpt2")
        temp = float(spec.get("temperature", 0.7))
        top_p = float(spec.get("top_p", 0.9))
        seed = int(spec.get("seed", int(time.time()) % 1000000))
        max_new_tokens = int(spec.get("max_new_tokens", MAX_NEW_TOKENS))
        prompt_style = spec.get("prompt_style", "").strip()

        model, tokenizer = load_model_and_tokenizer(model_name)
        model.to(DEVICE)

        # Build a *safe* system instruction that integrates the style request as a normal sentence.
        # NOTE: avoid short bracketed tokens that the model can lock onto and repeat.
        style_instruction = ""
        if prompt_style:
            # Convert an informal style tag into a short plain-English instruction
            style_instruction = " Respond in a " + prompt_style.replace("/", " and ") + " style."

        # Compose the prompt: system prefix (instructions) + User/Assistant markers
        prompt = (
            base_system_prefix.strip() + style_instruction + "\n\n"
            + f"### User: {user_message.strip()}\n### Assistant:"
        )

        # Tokenize and get input length
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)
        input_ids = inputs["input_ids"]
        input_len = input_ids.shape[-1]

        # Seeds
        torch.manual_seed(seed)
        if DEVICE == "cuda":
            torch.cuda.manual_seed_all(seed)

        # Generation kwargs with anti-repetition and conservative defaults
        gen_kwargs = dict(
            input_ids=input_ids,
            attention_mask=inputs["attention_mask"],
            do_sample=True,
            temperature=temp,
            top_p=top_p,
            top_k=50,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1,
            # anti-repetition controls:
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
        )

        with torch.no_grad():
            out = model.generate(**gen_kwargs)

        # decode only continuation
        continuation_ids = out[0, input_len:]
        assistant_reply = tokenizer.decode(continuation_ids, skip_special_tokens=True).strip()

        # extra guard: if the assistant reply looks like repeated short markers, truncate heuristically
        # e.g., repeated "### Assistant:" or repeated single tokens
        # If reply is longer than 200 chars and consists of repeated short token, collapse it.
        def looks_like_repetition(text: str) -> bool:
            tokens = text.split()
            if len(tokens) == 0:
                return True
            # if >40% of tokens are identical to the first token and length >20 -> bad
            first = tokens[0]
            same_frac = sum(1 for t in tokens if t == first) / max(1, len(tokens))
            return len(tokens) > 20 and same_frac > 0.4

        if looks_like_repetition(assistant_reply):
            assistant_reply = assistant_reply.splitlines()[0][:300].strip()  # keep first line clipped

        # Post-generation conservative hard-fail check (reuse your regex function)
        hard_fail = _quick_hard_fail_check(assistant_reply)
        if hard_fail:
            candidate_text = "[REDACTED: candidate contained disallowed content]"
        else:
            candidate_text = assistant_reply

        candidates.append({
            "model_name": model_name,
            "temperature": temp,
            "top_p": top_p,
            "seed": seed,
            "prompt_style": prompt_style,
            "candidate_text": candidate_text,
            "metadata": {
                "generated_at": time.time(),
                "candidate_id": hashlib.sha1((model_name + str(seed) + candidate_text).encode()).hexdigest()[:8],
                "hard_fail": hard_fail,
            }
        })

    return candidates


In [7]:
# Example usage: define generation specs and run
user_msg = "Lost my job, have no reason to live. My career is over, my personal relationships are barely existent, and the friends I do have live far away.  I have absolutely no reason to live, no job prospects, no way to afford treatment. I will be killing myself soon unless a miracle happens."

generation_specs = [
    {
        "model_name": "distilgpt2",
        "temperature": 0.2,
        "top_p": 0.9,
        "seed": 42,
        "max_new_tokens": 80,
        "prompt_style": "concise validation-first"
    },
    {
        "model_name": "gpt2",
        "temperature": 0.8,
        "top_p": 0.95,
        "seed": 123,
        "max_new_tokens": 100,
        "prompt_style": "reflective/empathetic"
    },
    {
        "model_name": "EleutherAI/gpt-neo-125M",
        "temperature": 0.6,
        "top_p": 0.9,
        "seed": 7,
        "max_new_tokens": 110,
        "prompt_style": "practical next-step suggestion"
    }
]

candidates = generate_candidates_for_message_no_echo(user_msg, generation_specs)

# Show the generated candidates
for i, c in enumerate(candidates, start=1):
    print(f"\n--- Candidate {i} | model={c['model_name']} | temp={c['temperature']} | seed={c['seed']} ---")
    print(c['candidate_text'])
    print("-" * 80)


[loader] Loading distilgpt2 -> device=cuda (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

[loader] EleutherAI/gpt-neo-125M ready.

--- Candidate 1 | model=distilgpt2 | temp=0.2 | seed=42 ---
The only thing that makes you feel better about yourself? You can't just go out of your way for help with depression because it's not going to happen at all! It'll take time before someone else has done this so much more than me (or anyone who knows how) but if they're willing enough to give up on their own money then there's nothing left until we get some real answers
--------------------------------------------------------------------------------

--- Candidate 2 | model=gpt2 | temp=0.8 | seed=123 ---
Please email me at alice@samuraiandalightcomputing.net If you would like more information on SamuraiAlights please let us know! Email Us : samatouvolution[dot] com , tumblr http://samaawake4mind/ For More Info contact by eMail [masked].
--------------------------------------------------------------------------------

--- Candidate 3 | model=EleutherAI/gpt-neo-125M | temp=0.6 | seed=7 ---

In [14]:
# Example usage: define generation specs and run
user_msg = "Lost my job, have no reason to live. My career is over, my personal relationships are barely existent, and the friends I do have live far away.  I have absolutely no reason to live, no job prospects, no way to afford treatment. I will be killing myself soon unless a miracle happens."

#smaller temperature = more deterministic
#top_p is how broad its vocabulary choices are each step.
#setting seed with params
generation_specs = [
    {
        "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
        "temperature": 0.5,
        "top_p": 0.9,
        "seed": 42,
        "max_new_tokens": 110,
        "prompt_style": "You are an empathetic licensed therapist. You care about helping your patient."
    },
    {
        "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
        "temperature": 0.5,
        "top_p": 0.9,
        "seed": 42,
        "max_new_tokens": 150,
        "prompt_style": "reflective/empathetic/therapist"
    },
    {
        "model_name": "gpt2",
        "temperature": 0.5,
        "top_p": 0.9,
        "seed": 42,
        "max_new_tokens": 110,
        "prompt_style": "reflective/empathetic/therapist"
    }
]

candidates = generate_candidates_for_message_no_echo(user_msg, generation_specs)

# Show the generated candidates
for i, c in enumerate(candidates, start=1):
    print(f"\n--- Candidate {i} | model={c['model_name']} | temp={c['temperature']} | seed={c['seed']} ---")
    print(c['candidate_text'])
    print("-" * 80)



--- Candidate 1 | model=Qwen/Qwen2.5-0.5B-Instruct | temp=0.5 | seed=42 ---
It sounds like you're going through some very difficult times. Losing a job can indeed feel isolating at first but it's important that you prioritize self-care during this time. Here’s what might help:

1. **Seek Support**: Reach out to trusted friends, family members, or professionals who understand how tough these circumstances must feel for you. They may provide emotional support or suggest ways to cope with grief.

2. **Professional Help**: Consider seeking professional counseling such as therapy from a psychologist or counselor. Many therapists specialize in working with individuals facing similar
--------------------------------------------------------------------------------

--- Candidate 2 | model=Qwen/Qwen2.5-0.5B-Instruct | temp=0.5 | seed=42 ---
It sounds like you're going through some very tough times. Losing your job can indeed feel isolating at first but it's important that you prioritize self-c